# Frontieres-10 — Neural diving : un plongeur appris pour CP-SAT

## Du conseil aux valeurs aux branches prouvées : ce que le diving apporte, mesure à l'appui

[← Partie 5](README.md) | [↑ Search](../README.md) | [<< App-28 Learning to Branch](Frontieres-06-LearningToBranch-Generalization-Audit-CSharp.ipynb) | [>> Frontieres-11 Abstraction à dette](Frontieres-11-Abstraction-Dette-ShortestPath-Python.ipynb)

> **Durée estimée : 60 minutes**

## Hommage et question scientifique

Une branche de la recherche « machine learning for combinatorial optimization » promet d'apprendre une **solution initiale** pour accélérer le solveur : c'est le **diving** de Nair et al. (2021), *« Solving Mixed Integer Programs Using Neural Networks »* (arXiv:2012.13349, v3, juillet 2021). Leur apport se compose de deux gestes distincts : la **priorité de branchement** (choisir quelle variable contribuer en premier, à chaque nœud) et le **plongement** (apprendre quel *sous-ensemble de variables fixer tout de suite*). La composante branchement a déjà été auditée dans ce dépôt : [App-28 Learning to Branch](Frontieres-06-LearningToBranch-Generalization-Audit-CSharp.ipynb) y mesure qu'une politique locale fidèle ne garantit ni un arbre plus petit ni un solveur plus rapide. Ce notebook reprend l'autre composante, sur un terrain contrôlé : **un plongeur MLP qui prédit une affectation complète ; les exercices proposent un conseil partiel, injectée comme `hint` (conseil) dans un solveur réel, OR-Tools CP-SAT, sur une famille de coloration de graphe calibrée pour brancher.**

La reproduction est entièrement locale, sur une famille synthétique de taille notebook, et ne copie ni code ni figure de l'article. Le but n'est pas de « refaire l'article », mais de mesurer sur un cas où la recherche travaille vraiment ce que le mécanisme promet : moins de nœuds, ou une recherche plus stable. Verdict au programme : **deux nombres ensemble** — la médiane des branches, et la queue de distribution.

**Prérequis :** C#, CP-SAT natif 9.15.6755, Python 3.13 avec scikit-learn et NumPy via Python.NET. Les graphes et matrices sont des entrées reproduites depuis les générateurs Python originaux, sans résultats préenregistrés.

## 1. Où le diving s'insère dans l'architecture d'un solveur

Un solveur de programmation par contraintes (CP-SAT ici) cherche en alternant deux opérations : **brancher** (fixer une variable à une valeur) et **propager** (déduire les conséquences). La taille de la recherche se mesure en nombre de branches explorées avant la première preuve d'optimalité (`NumBranches()`).

L'apprentissage à base de solutions a deux levers bien distincts, souvent mélangés :

| Levier | Portée | Effet documenté |
|---|---|---|
| **Branching appris** | une décision à *chaque nœud* (quelle variable) | imité localement, ne garantit pas un arbre plus petit (Frontieres-06) |
| **Diving appris** | une décision *une fois* (quelles valeurs suggérer) | donne un point de départ : la recherche contient-elle des mauvaises décisions précoces ? |

Le diving ne change pas la stratégie de branche, il change l'**état initial**. Dans CP-SAT, il se traduit par `AddHint(var, valeur)` : le solveur reçoit un conseil, s'en sert pour diriger sa recherche, et le **corrige** si le conseil contredit les contraintes. Cette correction est mesurable : quand un hint viole des arêtes d'adjacence, on dit qu'il porte des **conflits**.

In [1]:
// CP-SAT natif pour l'optimisation, Python.NET uniquement pour le MLP.
#r "nuget: Google.OrTools, 9.15.6755"
#r "nuget: pythonnet, 3.1.0"
#load "../../Config/PythonNetLoader.cs"
using System;
using System.Globalization;
using Python.Runtime;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
PythonNetLoader.Initialize(typeof(PythonEngine).Assembly, "sklearn");
using (Py.GIL())
{
    dynamic sk = Py.Import("sklearn");
    Console.WriteLine($"scikit-learn {sk.__version__} ; CP-SAT {typeof(Google.OrTools.Sat.CpSolver).Assembly.GetName().Version}");
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Google.OrTools, 9.15.6755 pythonnet, 3.1.0

scikit-learn 1.9.1 ; CP-SAT 9.15.6755.0


## 2. Un solveur coloriable, instrumenté, paramétrable

Le modèle est volontairement simple : une **coloration propre** à 6 couleurs — chaque sommet reçoit exactement une couleur (`AddExactlyOne`), deux sommets adjacents ne partagent aucune couleur (`c[v, j] + c[u, j] <= 1`), et le solveur minimise le nombre de couleurs réellement utilisées (variables `used`). Le nombre optimal est calculé pour chaque graphe.

Le paramètre important est `hint` : `null` pour la recherche pure, ou une matrice one-hot (60 × 6) de valeurs suggérées. Les suggestions sont exprimées par `model.AddHint(...)` — le solveur s'en sert pour orienter sa recherche **sans obéir** : il peut les corriger. Chaque résolution rapporte `NumBranches()` et le temps mur.

In [2]:
// Coloration native CP-SAT : les hints ne sont pas des contraintes et ne prouvent pas leur faisabilité.
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using System.Text.Json;
using Google.OrTools.Sat;
using Python.Runtime;

record ColorResult(CpSolverStatus Status, int Colors, long Branches, double Seconds, int[][] Solution);
record CoverResult(CpSolverStatus Status, long Branches, double Seconds, double Objective);
class NeuralBench
{
    public const int NV = 60, KMAX = 6;
    public JsonElement Data;
    public NeuralBench(JsonElement data)
    {
        Data = data;
    }

    // Les deux orientations de chaque arête alimentent le modèle et le validateur avec le même graphe.
    public bool[, ] Graph(int seed)
    {
        var adj = new bool[NV, NV];
        foreach (var e in Data.GetProperty("graphs").GetProperty(seed.ToString()).EnumerateArray())
        {
            int v = e[0].GetInt32(), u = e[1].GetInt32();
            adj[v, u] = adj[u, v] = true;
        }

        return adj;
    }

    public int[][] Cover(string regime, int seed) => Data.GetProperty("covers").GetProperty(regime).GetProperty(seed.ToString()).EnumerateArray().Select(row => row.EnumerateArray().Select(x => x.GetInt32()).ToArray()).ToArray();
    // Le sommet choisit une couleur et le modèle minimise le nombre de couleurs déclarées actives.
    public ColorResult SolveColor(bool[, ] adj, int[][] hint = null, double seconds = 15)
    {
        var model = new CpModel();
        var c = new BoolVar[NV, KMAX];
        for (int v = 0; v < NV; v++)
            for (int j = 0; j < KMAX; j++)
                c[v, j] = model.NewBoolVar($"c{v}_{j}");
        for (int v = 0; v < NV; v++)
            model.AddExactlyOne(Enumerable.Range(0, KMAX).Select(j => c[v, j]));
        for (int v = 0; v < NV; v++)
            for (int u = v + 1; u < NV; u++)
                if (adj[v, u])
                    for (int j = 0; j < KMAX; j++)
                        model.Add(c[v, j] + c[u, j] <= 1);
        var used = Enumerable.Range(0, KMAX).Select(j => model.NewBoolVar($"u{j}")).ToArray();
        for (int j = 0; j < KMAX; j++)
            for (int v = 0; v < NV; v++)
                model.Add(used[j] >= c[v, j]);
        model.Minimize(LinearExpr.Sum(used));
        if (hint != null)
            for (int v = 0; v < NV; v++)
                for (int j = 0; j < KMAX; j++)
                    // Les zéros font partie du conseil binaire pour une ligne fournie.
                    model.AddHint(c[v, j], hint[v][j]);
        var solver = new CpSolver
        {
            StringParameters = $"max_time_in_seconds:{seconds} random_seed:7 num_search_workers:1"};
        var sw = Stopwatch.StartNew();
        var status = solver.Solve(model);
        sw.Stop();
        if (status != CpSolverStatus.Optimal)
            return new(status, 0, solver.NumBranches(), sw.Elapsed.TotalSeconds, null);
        var sol = Enumerable.Range(0, NV).Select(v => Enumerable.Range(0, KMAX).Select(j => (int)solver.Value(c[v, j])).ToArray()).ToArray();
        int colors = Enumerable.Range(0, KMAX).Count(j => sol.Any(row => row[j] == 1));
        return new(status, colors, solver.NumBranches(), sw.Elapsed.TotalSeconds, sol);
    }

    // Le contrôle indépendant examine les affectations renvoyées et les conflits sur les arêtes.
    public static bool Valid(bool[, ] adj, int[][] sol)
    {
        if (sol == null || sol.Length != NV || sol.Any(row => row.Length != KMAX || row.Count(v => v == 1) != 1 || row.Any(v => v != 0 && v != 1)))
            return false;
        for (int v = 0; v < NV; v++)
            for (int u = v + 1; u < NV; u++)
                if (adj[v, u] && Array.IndexOf(sol[v], 1) == Array.IndexOf(sol[u], 1))
                    return false;
        return true;
    }

    // Réindexer les couleurs selon leur première apparition réduit la symétrie des labels.
    public static int[][] Canonize(int[][] sol)
    {
        var order = new List<int>();
        var result = new int[NV][];
        for (int v = 0; v < NV; v++)
        {
            int old = Array.IndexOf(sol[v], 1);
            if (!order.Contains(old))
                order.Add(old);
            result[v] = new int[KMAX];
            result[v][order.IndexOf(old)] = 1;
        }

        return result;
    }

    public static int Conflicts(bool[, ] adj, int[][] sol)
    {
        int count = 0;
        for (int v = 0; v < NV; v++)
            for (int u = v + 1; u < NV; u++)
                if (adj[v, u] && Array.IndexOf(sol[v], 1) == Array.IndexOf(sol[u], 1))
                    count++;
        return count;
    }

    public CoverResult SolveCover(int[][] rows, int n, int[] hint = null, double seconds = 10)
    {
        var m = new CpModel();
        var x = Enumerable.Range(0, n).Select(i => m.NewBoolVar($"x{i}")).ToArray();
        foreach (var row in rows)
            m.Add(LinearExpr.Sum(row.Select(j => (LinearExpr)x[j])) >= 2);
        m.Minimize(LinearExpr.Sum(x));
        if (hint != null)
            for (int j = 0; j < n; j++)
                m.AddHint(x[j], hint[j]);
        var solver = new CpSolver
        {
            StringParameters = $"max_time_in_seconds:{seconds} random_seed:7 num_search_workers:1"};
        var sw = Stopwatch.StartNew();
        var st = solver.Solve(m);
        sw.Stop();
        return new(st, solver.NumBranches(), sw.Elapsed.TotalSeconds, solver.ObjectiveValue);
    }

    public static int[] Greedy(int[][] rows, int n)
    {
        var chosen = new int[n];
        var cover = new int[rows.Length];
        // Chaque ligne exige deux colonnes choisies. Le gain glouton compte les demandes encore ouvertes.
        while (cover.Any(x => x < 2))
        {
            int best = -1, gain = -1;
            for (int j = 0; j < n; j++)
            {
                if (chosen[j] != 0)
                    continue;
                int g = Enumerable.Range(0, rows.Length).Count(i => cover[i] < 2 && rows[i].Contains(j));
                if (g > gain)
                {
                    gain = g;
                    best = j;
                }
            }

            if (best < 0 || gain <= 0)
                throw new InvalidOperationException("Couverture double impossible");
            chosen[best] = 1;
            for (int i = 0; i < rows.Length; i++)
                if (rows[i].Contains(best))
                    cover[i]++;
        }

        return chosen;
    }

    public CoverResult SolveKnapsack()
    {
        var k = Data.GetProperty("knapsack");
        var w = k.GetProperty("weights").EnumerateArray().Select(r => r.EnumerateArray().Select(x => x.GetInt32()).ToArray()).ToArray();
        var profit = k.GetProperty("profits").EnumerateArray().Select(x => x.GetInt32()).ToArray();
        var model = new CpModel();
        var x = Enumerable.Range(0, profit.Length).Select(i => model.NewBoolVar($"x{i}")).ToArray();
        foreach (var row in w)
            model.Add(LinearExpr.WeightedSum(x, row.Select(v => (long)v).ToArray()) <= row.Sum() / 2);
        model.Maximize(LinearExpr.WeightedSum(x, profit.Select(v => (long)v).ToArray()));
        var solver = new CpSolver
        {
            StringParameters = "max_time_in_seconds:3 random_seed:7 num_search_workers:1"
        };
        var sw = Stopwatch.StartNew();
        var status = solver.Solve(model);
        sw.Stop();
        return new(status, solver.NumBranches(), sw.Elapsed.TotalSeconds, solver.ObjectiveValue);
    }

    public static double[] Flatten(bool[, ] adj)
    {
        var a = new double[NV * NV];
        for (int i = 0; i < NV; i++)
            for (int j = 0; j < NV; j++)
                a[i * NV + j] = adj[i, j] ? 1 : 0;
        return a;
    }
}

var input = JsonDocument.Parse(File.ReadAllText("data/frontieres-learned-inputs/neural-inputs.json"));
var bench = new NeuralBench(input.RootElement);
var adj0 = bench.Graph(5000);
var smoke = bench.SolveColor(adj0);
Console.WriteLine($"Graph 5000 : statut={smoke.Status}, couleurs={smoke.Colors}, branches={smoke.Branches}, valide={NeuralBench.Valid(adj0, smoke.Solution)}");

Graph 5000 : statut=Optimal, couleurs=4, branches=2738, valide=True


### La symétrie des couleurs : un label en cache un autre

Toute solution peut être réécrite en **permutant les noms des couleurs** : colorier le sommet A en rouge et B en bleu, ou A en bleu et B en rouge, est *la même* affectation du point de vue des contraintes. Le solveur peut donc rendre l'une des $6! = 720$ réécritures équivalentes.

Pour un apprentissage supervisé de la solution (notre plongeur prédit un one-hot par sommet), cette symétrie est un **bruit d'étiquettes** : deux exemples identiques du point de vue du problème reçoivent des labels différents. Avant d'entraîner quoi que ce soit, on **canonise** les solutions : on renomme les couleurs par ordre de première apparition le long des sommets — la première couleur rencontrée devient 0, la suivante 1, etc. La table de renommage est propre à chaque solution, elle ne normalise donc rien d'autre que le *nom* des couleurs.

In [3]:
// Une permutation des couleurs ne change pas le problème : la canonisation l'annule.
if (smoke.Solution == null)
    throw new InvalidOperationException("La solution du contrôle est requise");
var permutation = input.RootElement.GetProperty("permutation_123").EnumerateArray().Select(x => x.GetInt32()).ToArray();
var permuted = smoke.Solution.Select(row =>
{
    var result = new int[NeuralBench.KMAX];
    result[permutation[Array.IndexOf(row, 1)]] = 1;
    return result;
}).ToArray();
Console.WriteLine($"Canonisation invariante : {NeuralBench.Canonize(smoke.Solution).SelectMany(x => x).SequenceEqual(NeuralBench.Canonize(permuted).SelectMany(x => x))}");

Canonisation invariante : True


## 3. Calibrer la famille : la fenêtre où la recherche travaille

Comparer un hint suppose que la recherche travaille. Les contrôles ci-dessous résolvent une couverture double creuse et un knapsack multidimensionnel corrélé avec le même solveur et le même budget que la source. La coloration est contrôlée par le graphe 5000 ; le cadran suivant balaie deux densités sur vingt graines, avec et sans hint glouton.

In [4]:
// Deux contrôles de calibration : couverture double et knapsack corrélé.
var coverSmoke = bench.SolveCover(bench.Cover("presolve", 4001), 60, seconds: 3);
var knap = bench.SolveKnapsack();
Console.WriteLine($"Couverture double : statut={coverSmoke.Status}, branches={coverSmoke.Branches}, objectif={coverSmoke.Objective}");
Console.WriteLine($"Knapsack corrélé : statut={knap.Status}, branches={knap.Branches}, objectif={knap.Objective}");

Couverture double : statut=Optimal, branches=0, objectif=24


Knapsack corrélé : statut=Feasible, branches=19173, objectif=2958


La cellule suivante compare deux régimes sur vingt graines.

In [5]:
// Ajustement du MLP réel : le pont ne contient ni solveur ni logique de validation.
class MlpBridge : IDisposable
{
    readonly PyObject model, np;
    public MlpBridge(double[][] x, int[][] y)
    {
        using (Py.GIL())
        {
            dynamic sk = Py.Import("sklearn.neural_network");
            np = Py.Import("numpy");
            dynamic numpy = np;
            dynamic network = sk.MLPClassifier(hidden_layer_sizes: new PyTuple(new[] { new PyInt(128), new PyInt(128) }), max_iter: 3000, random_state: 0);
            dynamic xx = numpy.array(x);
            dynamic yy = numpy.array(y);
            network.fit(xx, yy);
            model = (PyObject)network;
        }
    }

    public double[] Predict(double[] features)
    {
        using (Py.GIL())
        {
            dynamic numpy = np;
            dynamic m = model;
            // Le MLP traite chaque position sommet-couleur comme une sortie binaire.
            dynamic values = m.predict_proba(numpy.array(new[] { features }));
            string text = values.tolist().ToString();
            using var doc = JsonDocument.Parse(text);
            return doc.RootElement[0].EnumerateArray().Select(x => x.GetDouble()).ToArray();
        }
    }

    public void Dispose()
    {
        using (Py.GIL())
        {
            model.Dispose();
            np.Dispose();
        }
    }
}

record NeuralRow(int Seed, int KA, int KB, long BranchA, long BranchB, double TimeA, double TimeB, int HintConflicts, bool ValidA, bool ValidB, string StatusA, string StatusB);
static double Med10(IEnumerable<double> values)
{
    var a = values.OrderBy(x => x).ToArray();
    return a.Length % 2 == 1 ? a[a.Length / 2] : (a[a.Length / 2 - 1] + a[a.Length / 2]) / 2;
}

static double Pearson(double[] a, double[] b)
{
    double ma = a.Average(), mb = b.Average();
    double numerator = Enumerable.Range(0, a.Length).Sum(i => (a[i] - ma) * (b[i] - mb));
    double da = Math.Sqrt(a.Sum(x => (x - ma) * (x - ma))), db = Math.Sqrt(b.Sum(x => (x - mb) * (x - mb)));
    return da == 0 || db == 0 ? double.NaN : numerator / (da * db);
}

var predictedProbabilities = new Dictionary<int, double[]>();
void CoverExperiments(NeuralBench bench)
{
    foreach (var(label, regime)in new[]
    {
        ("présolve", "presolve"),
        ("branchement", "branching")
    }

    )
    {
        var values = new List<(CoverResult Pure, CoverResult Hint)>();
        for (int seed = 4001; seed <= 4020; seed++)
        {
            // Une même matrice alimente les deux conditions de résolution pour cette graine.
            var mat = bench.Cover(regime, seed);
            var a = bench.SolveCover(mat, 60);
            var b = bench.SolveCover(mat, 60, NeuralBench.Greedy(mat, 60));
            values.Add((a, b));
        }

        Console.WriteLine($"{label} : pure médiane={Med10(values.Select(x => (double)x.Pure.Branches)):F0}, hint médiane={Med10(values.Select(x => (double)x.Hint.Branches)):F0}, optimum pur={values.Count(x => x.Pure.Status == CpSolverStatus.Optimal)}/20, optimum hint={values.Count(x => x.Hint.Status == CpSolverStatus.Optimal)}/20, même objectif={values.Count(x => x.Pure.Status == CpSolverStatus.Optimal && x.Hint.Status == CpSolverStatus.Optimal && x.Pure.Objective == x.Hint.Objective)}/20");
        Console.WriteLine($"  pur : min={values.Min(x => x.Pure.Branches)}, max={values.Max(x => x.Pure.Branches)}, zéro={values.Count(x => x.Pure.Branches == 0)}/20, temps médian={Med10(values.Select(x => x.Pure.Seconds)):F3} s");
        Console.WriteLine($"  hint : min={values.Min(x => x.Hint.Branches)}, max={values.Max(x => x.Hint.Branches)}, zéro={values.Count(x => x.Hint.Branches == 0)}/20, temps médian={Med10(values.Select(x => x.Hint.Seconds)):F3} s, moins={values.Count(x => x.Hint.Branches < x.Pure.Branches)}, plus={values.Count(x => x.Hint.Branches > x.Pure.Branches)}");
    }
}

List<NeuralRow> Experiment10(NeuralBench bench)
{
    var trainX = new List<double[]>();
    var trainY = new List<int[]>();
    var branches = new List<double>();
    var ks = new List<double>();
    for (int seed = 5000; seed < 5060; seed++)
    {
        var graph = bench.Graph(seed);
        var result = bench.SolveColor(graph);
        // Les labels du train exigent un optimum certifié et une affectation validée.
        if (result.Status != CpSolverStatus.Optimal || !NeuralBench.Valid(graph, result.Solution))
            throw new InvalidOperationException($"Entraînement non certifié : {seed} {result.Status}");
        trainX.Add(NeuralBench.Flatten(graph));
        // Le même ordre des sommets aplatit les labels après canonisation.
        trainY.Add(NeuralBench.Canonize(result.Solution).SelectMany(x => x).ToArray());
        ks.Add(result.Colors);
        branches.Add(result.Branches);
        Console.WriteLine($"Train {seed}: {result.Colors} couleurs, {result.Branches} branches");
    }

    Console.WriteLine($"Train : {trainX.Count}, couleurs médianes={Med10(ks)}, branches médianes={Med10(branches)}");
    var trainWatch = Stopwatch.StartNew();
    using var model = new MlpBridge(trainX.ToArray(), trainY.ToArray());
    trainWatch.Stop();
    Console.WriteLine($"MLP réel entraîné en {trainWatch.Elapsed.TotalSeconds:F3} s");
    var rows = new List<NeuralRow>();
    // Les graphes du test utilisent une plage de graines disjointe de l’entraînement.
    for (int seed = 6000; seed < 6025; seed++)
    {
        var graph = bench.Graph(seed);
        var infer = Stopwatch.StartNew();
        var prob = model.Predict(NeuralBench.Flatten(graph));
        // Conserver les scores bruts permet ensuite de comparer des variantes du hint.
        predictedProbabilities[seed] = prob;
        infer.Stop();
        if (prob.Length != NeuralBench.NV * NeuralBench.KMAX)
            throw new InvalidOperationException("Dimension modèle inattendue");
        var hint = Enumerable.Range(0, NeuralBench.NV).Select(v =>
        {
            var row = new int[NeuralBench.KMAX];
            int best = Enumerable.Range(0, NeuralBench.KMAX).OrderByDescending(j => prob[v * NeuralBench.KMAX + j]).First();
            row[best] = 1;
            return row;
        }).ToArray();
        var pure = bench.SolveColor(graph);
        var hinted = bench.SolveColor(graph, hint);
        // Le duel associe les deux statuts et validations au nombre de conflits du hint brut.
        var record = new NeuralRow(seed, pure.Colors, hinted.Colors, pure.Branches, hinted.Branches, pure.Seconds, hinted.Seconds, NeuralBench.Conflicts(graph, hint), NeuralBench.Valid(graph, pure.Solution), NeuralBench.Valid(graph, hinted.Solution), pure.Status.ToString(), hinted.Status.ToString());
        rows.Add(record);
        Console.WriteLine($"Test {seed}: A={pure.Colors}/{pure.Branches}, B={hinted.Colors}/{hinted.Branches}, conflits={record.HintConflicts}, inférence={infer.Elapsed.TotalSeconds:F4}s, valides={record.ValidA && record.ValidB}");
    }

    string output = Path.Combine(Path.GetTempPath(), "coursia-frontieres10-csharp");
    Directory.CreateDirectory(output);
    File.WriteAllText(Path.Combine(output, "diving_results.json"), JsonSerializer.Serialize(rows));
    Console.WriteLine($"Test : {rows.Count}, même objectif certifié={rows.Count(x => x.StatusA == "Optimal" && x.StatusB == "Optimal" && x.KA == x.KB)}/{rows.Count}, deux solutions valides={rows.Count(x => x.ValidA && x.ValidB)}/{rows.Count} ; résultats JSON générés.");
    return rows;
}

// Comparer 2 régimes, 20 graines, recherche pure et hint glouton.
CoverExperiments(bench);

présolve : pure médiane=525, hint médiane=421, optimum pur=20/20, optimum hint=20/20, même objectif=20/20


  pur : min=0, max=2223, zéro=1/20, temps médian=0.015 s


  hint : min=0, max=1659, zéro=1/20, temps médian=0.014 s, moins=12, plus=6


branchement : pure médiane=1275, hint médiane=1152, optimum pur=20/20, optimum hint=20/20, même objectif=20/20


  pur : min=329, max=5323, zéro=0/20, temps médian=0.400 s


  hint : min=636, max=3035, zéro=0/20, temps médian=0.392 s, moins=11, plus=9


### Cadran de régime

Comparer les médianes de branches, les statuts et les objectifs pour les deux densités. Un hint est un conseil : l'égalité des objectifs certifiés et la validité des solutions sont vérifiées séparément du nombre de branches.

## 4. Le plongeur : un MLP qui prédit l'affectation

Le prototype qui fonde ce notebook (mesures du 2026-09-23, seeds constants)

- **Train** : les graphes 5000..5059 (60 instances), chacun résolu à l'optimum, solutions **canonisées** ;
- **Features** : matrice d'adjacence aplatie (60 × 60 = 3600 booléens) ;
- **Labels** : one-hot (60 × 6) de la couleur canonique de chaque sommet, aplati ;
- **Modèle** : `MLPClassifier((128, 128))`, `max_iter=3000`, `random_state=0` — sur des données 100 % déterministes ;
- **Test** : graphes 6000..6024 (25 instances), non vus au train ; prédiction → one-hot par `argmax` sur les 6 sorties par sommet ;
- **Mesure A/B** : résoudre chaque graphe du test une fois **pur** (`hint=None`) et une fois **avec hint** (la prédiction canonisée) ; on compare `NumBranches()` et le temps mur. On compte aussi les **conflits** du hint : nombre d'arêtes dont les deux extrémités reçoivent la même couleur prédite.

Particularité de scikit-learn pour la sortie multioutput **binaire** (la version réellement utilisée est imprimée par le bilan de la section suivante, jamais figée ici) : `predict_proba` ne rend pas une liste de paires mais un tableau (1, 360) de probabilités de la classe 1 — on le reforme en (60, 6) et l'argmax par bloc garantit le one-hot.

In [6]:
// Les 60 graphes train produisent des labels optimaux avant l'entraînement réel.
var rows = Experiment10(bench);

Train 5000: 4 couleurs, 2738 branches


Train 5001: 4 couleurs, 2426 branches


Train 5002: 4 couleurs, 3029 branches


Train 5003: 4 couleurs, 2666 branches


Train 5004: 4 couleurs, 3058 branches


Train 5005: 4 couleurs, 2654 branches


... (78 outputs left out)

Test 6022: A=4/2743, B=4/2371, conflits=49, inférence=0.0040s, valides=True


Test 6023: A=4/2600, B=4/2241, conflits=44, inférence=0.0032s, valides=True


Test 6024: A=4/2904, B=4/2348, conflits=39, inférence=0.0026s, valides=True


Test : 25, même objectif certifié=25/25, deux solutions valides=25/25 ; résultats JSON générés.


### Exercice 1 — Tester un hint partiel *top-k*

Le banc exécuté utilise un hint complet. Cet exercice explore une variante qui ne suggère une couleur que pour les $k$ sommets dont le score de confiance $\max_j p_{v,j}$ est le plus élevé. Une suggestion reste facultative pour CP-SAT.

**Consigne** — Dans `SolveColor`, adaptez la boucle d’injection pour ignorer les lignes entièrement nulles. La boucle actuelle appelle `AddHint` pour chaque variable dès qu’une matrice est fournie, y compris pour les zéros. Complétez ensuite `HintTopK(probabilities, k)` pour produire une matrice `(60, 6)` dont les $k$ lignes sélectionnées sont one-hot et les autres entièrement nulles. Triez les sommets par confiance décroissante et départagez les égalités par indice de sommet. Vérifiez que les lignes non sélectionnées ne déclenchent aucun appel à `AddHint`. Dans `Experiment10`, mesurez `k = 10` et `k = 20` sur les mêmes graphes, puis comparez les branches et les temps au hint complet et au solveur pur.

In [7]:
// Exercice 1 : ne suggérer que les k sommets les plus confiants.
int[][] HintTopK(double[][] probabilities, int k)
{ /* TODO étudiant */
    return null;
}

Console.WriteLine(HintTopK(Enumerable.Range(0, NeuralBench.NV).Select(v => predictedProbabilities[6000].Skip(v * NeuralBench.KMAX).Take(NeuralBench.KMAX).ToArray()).ToArray(), 10) == null ? "Exercice à compléter" : "Hint partiel calculé");

Exercice à compléter


### Mesurer le contraste

Les indicateurs suivants comparent les deux recherches sur les mêmes graphes et séparent gain de branches et corrélation.

In [8]:
// Résumé des gains et contrôles de corrélation sur les 25 graphes test.
var certified = rows.Where(r => r.StatusA == "Optimal" && r.StatusB == "Optimal" && r.BranchA > 0).ToArray();
if (certified.Length > 0)
{
    var gain = certified.Select(r => (double)(r.BranchA - r.BranchB) / r.BranchA).ToArray();
    var pure = certified.Select(r => (double)r.BranchA).ToArray();
    var hint = certified.Select(r => (double)r.BranchB).ToArray();
    Console.WriteLine($"Médianes A/B={Med10(pure):F0}/{Med10(hint):F0} ; gain médian={Med10(gain):P1} ; améliorés={certified.Count(r => r.BranchB < r.BranchA)}/{certified.Length}");
    Console.WriteLine($"Pearson gain/A={Pearson(gain, pure):F3} ; gain/B={Pearson(gain, hint):F3}");
}

Médianes A/B=2829/2325 ; gain médian=15.1 % ; améliorés=25/25


Pearson gain/A=0.604 ; gain/B=0.015


### Exercice 2 — Projection faisable : réconcilier le hint avec les contraintes

Une projection gloutonne tente de produire un hint compatible avec les arêtes du graphe. Un sommet peut n’avoir aucune couleur libre dans la palette courante. Le coût de la projection et son effet sur la recherche se mesurent.

**Consigne** — Complétez `ProjectFeasible(hint, adjacency)` en travaillant sur une copie du hint. Pour une arête en conflit, cherchez chez l’un de ses sommets une couleur absente chez ses voisins. Fixez un budget de passages ou de réaffectations. Si les conflits subsistent et qu’aucune réaffectation légale ne progresse, ou si le budget est atteint, arrêtez et indiquez la cause, le budget consommé et le nombre de conflits résiduels. Pour une tentative réussie, validez le one-hot de chaque sommet et recomptez les conflits sur toutes les arêtes. Injectez seulement les projections validées. Consignez le taux de réussite et comparez branches et temps sur les mêmes graphes, en incluant le temps de projection.

In [9]:
// Exercice 2 : réduire à zéro les conflits entre arêtes.
int[][] ProjectFeasible(int[][] hint, bool[, ] adjacency)
{ /* TODO étudiant */
    return null;
}

var sampleAdj = bench.Graph(6000);
var sampleHint = Enumerable.Range(0, NeuralBench.NV).Select(v =>
{
    var row = new int[NeuralBench.KMAX];
    var p = predictedProbabilities[6000].Skip(v * NeuralBench.KMAX).Take(NeuralBench.KMAX).ToArray();
    row[Array.IndexOf(p, p.Max())] = 1;
    return row;
}).ToArray();
Console.WriteLine(ProjectFeasible(sampleHint, sampleAdj) == null ? "Exercice à compléter" : "Projection calculée");

Exercice à compléter


### Lecture du résultat

Les sorties ci-dessus comparent recherche pure et recherche avec hint sur les mêmes graphes tenus à l'écart. La réduction des branches, le coût du modèle, les conflits du conseil et les deux corrélations apportent des informations distinctes.

In [10]:
// Répartition et queue : une médiane ne révèle pas les cas difficiles.
foreach (var kind in new[]
{
    "pur",
    "hint"
}

)
{
    var a = rows.Select(r => (double)(kind == "pur" ? r.BranchA : r.BranchB)).OrderBy(x => x).ToArray();
    Console.WriteLine($"{kind} : médiane={Med10(a):F0}, moyenne={a.Average():F1}, maximum={a.Max():F0}");
}

foreach (var r in rows.OrderByDescending(r => r.BranchA).Take(3))
    Console.WriteLine($"graine={r.Seed}, A={r.BranchA}, B={r.BranchB}, conflits={r.HintConflicts}");

pur : médiane=2829, moyenne=2801.7, maximum=3291


hint : médiane=2325, moyenne=2349.4, maximum=2752


graine=6004, A=3291, B=2363, conflits=40


graine=6005, A=3274, B=2595, conflits=37


graine=6008, A=3184, B=2658, conflits=48


### Exercice 3 — Mesurer la variance, pas seulement la médiane

La section 4 oppose **médiane** et **max**. Mais le max ne résume pas la forme de la queue. La distribution des branches du solveur pur a une traîne lourde : combien d'instances dépassent 2× la médiane ? Et du côté hint, combien restent ?

**Consigne** : compléter `QueueTable(records)` pour qu'il rende une table avec, pour `BranchA` et `BranchB`, la médiane, l’écart-type et le nombre d’instances au-delà de deux fois la médiane de la série. Lire la différence de dispersion — c'est la signature du diving.

In [11]:
// Exercice 3 : comparer médiane, écart-type et compte supérieur à deux médianes.
object QueueTable(IEnumerable<NeuralRow> records)
{ /* TODO étudiant */
    return null;
}

Console.WriteLine(QueueTable(rows) == null ? "Exercice à compléter" : "Table calculée");

Exercice à compléter


## À retenir

Une prédiction de couleurs est un conseil, pas une preuve. La canonisation retire l'arbitraire des noms, tandis que CP-SAT vérifie les contraintes et le nombre de couleurs. Le contraste sur les mêmes instances sépare branches, temps, conflits et statut d'optimalité.

## Vérifiez votre compréhension

Pourquoi canoniser les couleurs avant l'entraînement ? Pourquoi un hint en conflit peut-il quand même être utile ?

<details><summary>Éléments de réponse</summary>

La permutation des noms représente la même coloration ; la canonisation réduit ce bruit de labels. Un hint n'est pas une contrainte : le solveur peut le corriger et prouver sa propre solution.
</details>

## Pour aller plus loin

- **Frontieres-06 Learning to Branch** ([Recherche](Frontieres-06-LearningToBranch-Generalization-Audit-Python.ipynb)) : l'autre moitié du geste Nair — la priorité de branchement apprise, auditée sur la même famille de questions (arbre × temps × coût d'inférence). La complémentarité des deux verdicts : aucune source d'amélioration locale ne se transforme en gain global sans mesure intégrée.
- **Exercices en extension** : remplacer le MLP par un classifieur basé sur la structure (features de degré), ou entraîner le plongeur sur les conflits (régression du nombre de conflits).

### Bibliographie

- Nair, V., Bartunov, S., Gimeno, F., et al. (2021). *Solving Mixed Integer Programs Using Neural Networks*. [arXiv:2012.13349](https://arxiv.org/abs/2012.13349).
- Bengio, Y., Lodi, A., Prouvost, A. (2021). *Machine Learning for Combinatorial Optimization: a Methodological Tour d’Horizon*. EJOR 290(2). [DOI 10.1016/j.ejor.2020.07.063](https://doi.org/10.1016/j.ejor.2020.07.063).
- Boussemart, F., Hemery, F., Lecoutre, C., Sais, L. (2004). *Boosting Systematic Search by Weighting Constraints* (dom/wdeg). ECAI. [Notice DBLP](https://dblp.org/rec/conf/ecai/BoussemartHLS04.html).